# 05 — Prepayment Model: PSA + Rate-Dependent CPR

Spec §6. Data source: `panel_logistic_2021_2025.parquet` (reuse; 2021–2025 originations).

This window is chosen specifically because the 2021 refinancing boom and 2022 CPR
collapse capture the widest rate variation available for characterizing prepayment.

Pipeline:
1. Load 2021-2025 panel, filter to FRM loans only
2. Compute scheduled principal and unscheduled prepayment (annuity formula)
3. Compute pool-level CPR time series
4. Fit PSA scalar speed k
5. Plot observed CPR vs. 100% PSA schedule
6. Demonstrate rate-dependent CPR responds correctly to refinancing incentive

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path('..') / 'src'))
import prepayment as pp

PROCESSED = Path('..') / 'data' / 'processed'
FIGURES   = Path('..') / 'artifacts' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load panel — FRM loans only

In [ ]:
panel = pd.read_parquet(PROCESSED / 'panel_logistic_2021_2025.parquet')

# Filter to fixed-rate mortgages only (CPR formula is undefined for ARMs)
frm = panel[panel['amortization_type'] == 'FRM'].copy()
print(f'Full panel:  {len(panel):,} rows')
print(f'FRM only:    {len(frm):,} rows ({len(frm)/len(panel):.1%} of panel)')
print(f'Unique loans: {frm.loan_id.nunique():,}')

## 2. Compute scheduled principal and prepayment (spec §6.2)

In [ ]:
frm = pp.compute_scheduled_principal(frm)
print(f'Columns added: bal_prev, scheduled_principal, prepayment')
print(f'\nSanity checks:')
print(f'  Negative scheduled_principal: {(frm.scheduled_principal < 0).sum():,} (expect 0)')
print(f'  Negative prepayment: {(frm.prepayment < 0).sum():,} (expect 0)')
total_prepay = frm['prepayment'].sum()
total_sched  = frm['scheduled_principal'].sum()
print(f'  Total prepayment: ${total_prepay/1e9:.2f}B')
print(f'  Total scheduled principal: ${total_sched/1e9:.2f}B')
print(f'  Prepay/Sched ratio: {total_prepay/total_sched:.2f}x')

## 3. Pool-level CPR time series

In [ ]:
monthly = pp.pool_cpr(frm)
monthly['date'] = pd.to_datetime(monthly['reporting_period'].astype(str), format='%Y%m')
monthly = monthly.sort_values('date')

print(f'Monthly CPR series: {len(monthly)} periods')
print(f'CPR range: {monthly.cpr.min():.4f} – {monthly.cpr.max():.4f}')
print(f'Mean CPR: {monthly.cpr.mean():.4f} ({monthly.cpr.mean()*100:.2f}%)')

## 4. Fit PSA scalar speed (spec §6.3)

In [ ]:
psa_k = pp.fit_psa_speed(monthly)
print(f'Pool equivalent PSA speed: {psa_k*100:.1f}% PSA')
print(f'(100% PSA = benchmark; >100% = faster than benchmark prepayment)')

## 5. Plot observed CPR vs. PSA schedule

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: time series of observed pool CPR
axes[0].plot(monthly['date'], monthly['cpr'] * 100, label='Observed CPR', linewidth=1.5)
axes[0].set_title('Pool-Level CPR (2021-2025)')
axes[0].set_xlabel('Reporting Period')
axes[0].set_ylabel('CPR (%)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend()

# Right: observed CPR vs. PSA schedules at various speeds
months_seq = np.arange(1, len(monthly) + 1)
for speed, color in [(0.5, 'green'), (1.0, 'blue'), (psa_k, 'red'), (2.0, 'orange')]:
    label = f'{speed*100:.0f}% PSA' + (' (fitted)' if abs(speed - psa_k) < 0.01 else '')
    axes[1].plot(
        months_seq,
        pp.psa_cpr_schedule(months_seq, speed) * 100,
        label=label, linestyle='--' if speed != psa_k else '-',
        color=color, linewidth=1.5
    )
axes[1].plot(months_seq[:len(monthly)], monthly['cpr'].values * 100,
             'k.', markersize=3, alpha=0.5, label='Observed')
axes[1].set_title('Observed CPR vs. PSA Schedules')
axes[1].set_xlabel('Month Index')
axes[1].set_ylabel('CPR (%)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / 'prepayment_cpr_vs_psa.png', bbox_inches='tight')
plt.show()

## 6. Rate-dependent CPR: verify response to refinancing incentive (spec §6.4)

Higher RI (borrower coupon > market rate) → more refinancing → higher CPR.  
At RI = 0 (at-the-money), pool prepays at approximately 100% PSA.

In [ ]:
ri_range = np.linspace(-0.03, 0.03, 100)  # -300bp to +300bp
k_vals   = pp.rate_dependent_psa_k(ri_range)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PSA speed vs. refinancing incentive
axes[0].plot(ri_range * 100, k_vals, 'steelblue', linewidth=2)
axes[0].axhline(1.0, color='k', linestyle='--', alpha=0.5, label='100% PSA')
axes[0].axvline(0, color='gray', linestyle=':', alpha=0.5, label='RI = 0 (at-market)')
axes[0].set_xlabel('Refinancing Incentive (bp)')
axes[0].set_ylabel('PSA Speed Multiplier k')
axes[0].set_title('Rate-Dependent PSA Speed\nk(RI) = k_min + (k_max - k_min)·σ(α + β·RI)')
axes[0].legend()

# Right: resulting CPR at different rate scenarios for a 6.5% coupon pool
coupon = 0.065
for market_rate, label in [(0.04, '4% market (refi boom)'), (0.065, '6.5% (at-money)'),
                            (0.08, '8% market (rate lock-in)')]:
    months_sim = np.arange(1, 361)
    ri_scenario = coupon - market_rate
    k_scenario = pp.rate_dependent_psa_k(np.array([ri_scenario]))[0]
    cpr_sim = pp.psa_cpr_schedule(months_sim, k_scenario)
    axes[1].plot(months_sim, cpr_sim * 100, label=f'{label} (k={k_scenario:.2f})', linewidth=1.5)

axes[1].set_title('Simulated CPR Schedules at Different Market Rates\n(6.5% coupon pool)')
axes[1].set_xlabel('Loan Age (months)')
axes[1].set_ylabel('CPR (%)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / 'prepayment_rate_dependent_cpr.png', bbox_inches='tight')
plt.show()

print('\nVerification: higher RI → higher CPR')
for market_rate in [0.04, 0.065, 0.08]:
    ri = coupon - market_rate
    k_val = pp.rate_dependent_psa_k(np.array([ri]))[0]
    print(f'  market={market_rate*100:.1f}%  RI={ri*100:+.1f}bp  k={k_val:.3f}  plateau CPR={k_val*0.06*100:.2f}%')